In [22]:
import os

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [23]:
import torch
import torch.nn as nn

In [24]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [25]:
device

device(type='cuda')

In [26]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.embed_size = embed_size
        self.num_heads = num_heads
        self.head_dim = embed_size // num_heads
        assert embed_size % num_heads == 0, "embed_size должен делиться на num_heads!"
        self.query = nn.Linear(embed_size, embed_size)
        self.key = nn.Linear(embed_size, embed_size)
        self.value = nn.Linear(embed_size, embed_size)
        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, queries, keys, values, mask=None):
        q = self.query(queries)
        k = self.key(keys)
        v = self.value(values)
        batch_size = queries.shape[0]
        query_len = queries.shape[1]
        key_len = keys.shape[1]
        q = q.reshape(batch_size, query_len, self.num_heads, self.head_dim)
        k = k.reshape(batch_size, key_len, self.num_heads, self.head_dim)
        v = v.reshape(batch_size, key_len, self.num_heads, self.head_dim)
        scores = (q.permute(0, 2, 1, 3) @ k.permute(0, 2, 3, 1)) / (self.head_dim ** 0.5)
        # mask = torch.tril(torch.ones(seq_len, seq_len))
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-1e20'))
        scores = self.fc_out((scores.softmax(dim=-1) @ v.permute(0, 2, 1, 3)).permute(0, 2, 1, 3).flatten(start_dim=2))
        return scores

In [28]:
class EncoderBlock(nn.Module):
    def __init__(self, embed_size, num_heads, forward_expansion=4):
        super(EncoderBlock, self).__init__()
        self.attention = MultiHeadAttention(embed_size, num_heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)
        self.feed_forward = nn.Sequential(nn.Linear(embed_size, forward_expansion * embed_size), nn.ReLU(), nn.Linear(forward_expansion * embed_size, embed_size))
    def forward(self, x, src_mask):
        out = self.norm1(x + self.attention(queries=x, keys=x,values=x, mask=src_mask))
        out = self.norm2(out + self.feed_forward(out))
        return out

In [29]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_size, num_heads, forward_expansion=4):
        super(DecoderBlock, self).__init__()
        self.attention = MultiHeadAttention(embed_size, num_heads)
        self.cross_attention = MultiHeadAttention(embed_size, num_heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)
        self.norm3 = nn.LayerNorm(embed_size)
        self.feed_forward = nn.Sequential(nn.Linear(embed_size, forward_expansion * embed_size), nn.ReLU(), nn.Linear(forward_expansion * embed_size, embed_size))
    def forward(self, x, enc_out, src_mask, trg_mask):
        x = self.norm1(x + self.attention(queries=x, keys=x, values=x, mask=trg_mask))
        x = self.norm2(x + self.cross_attention(queries=x, keys=enc_out, values=enc_out, mask=src_mask))
        x = self.norm3(x + self.feed_forward(x))
        return x

In [30]:
class FullTransformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, max_length, embed_size, num_heads, num_layers, src_pad_idx, trg_pad_idx, forward_expansion=4):
        super(FullTransformer, self).__init__()
        self.max_length = max_length
        self.embed_size = embed_size
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.forward_expansion = forward_expansion
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.src_word_embedding = nn.Embedding(src_vocab_size, embed_size)
        self.trg_word_embedding = nn.Embedding(trg_vocab_size, embed_size)
        self.position_embedding = nn.Embedding(max_length, embed_size)
        self.encoder = nn.ModuleList([EncoderBlock(embed_size, num_heads, forward_expansion) for _ in range(num_layers)])
        self.decoder = nn.ModuleList([DecoderBlock(embed_size, num_heads, forward_expansion) for _ in range(num_layers)])
        self.fc_out = nn.Linear(embed_size, trg_vocab_size)
    def make_src_mask(self, src):
        # src shape: (batch_size, src_len)
        # Ставим True там, где слово НЕ является отступом (<PAD>)
        # Добавляем размерности unsqueeze, чтобы маска подошла под 4D тензор внимания (batch, heads, seq, seq)
        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)
        # return shape: (batch_size, 1, 1, src_len)
        return src_mask

    def make_trg_mask(self, trg):
        # trg shape: (batch_size, trg_len)
        batch_size, trg_len = trg.shape

        trg_pad_mask = (trg != self.trg_pad_idx).unsqueeze(1).unsqueeze(2)

        trg_causal_mask = torch.tril(torch.ones((trg_len, trg_len))).type(torch.bool).to(trg.device)

        trg_mask = trg_pad_mask & trg_causal_mask
        return trg_mask
    def forward(self, src, trg):
        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)
        batch_size = src.shape[0]
        src_len = src.shape[1]
        trg_len = trg.shape[1]
        positions_src = torch.arange(0, src_len).expand(batch_size, src_len).to(src.device)
        positions_trg = torch.arange(0, trg_len).expand(batch_size, trg_len).to(trg.device)
        positions_embed_src = self.position_embedding(positions_src)
        positions_embed_trg = self.position_embedding(positions_trg)
        embed_src = self.src_word_embedding(src)
        embed_trg = self.trg_word_embedding(trg)
        src_out = embed_src + positions_embed_src
        trg_out = embed_trg + positions_embed_trg
        for layer in self.encoder:
            src_out = layer(src_out, src_mask)
        for layer in self.decoder:
            trg_out = layer(trg_out, src_out, src_mask, trg_mask)
        out = self.fc_out(trg_out)
        return out
        



In [31]:
import torch.optim as optim

# Задаем тестовые гиперпараметры
src_vocab_size = 5000
trg_vocab_size = 5000
src_pad_idx = 0
trg_pad_idx = 0

# 1. Создаем нашу модель
model = FullTransformer(
    src_vocab_size=src_vocab_size,
    trg_vocab_size=trg_vocab_size,
    src_pad_idx=src_pad_idx,
    trg_pad_idx=trg_pad_idx,
    embed_size=256,      # Размер скрытых представлений
    num_heads=8,         # Число голов внимания
    num_layers=3,        # Количество слоёв
    max_length=100,
    forward_expansion=4
)

# 2. Оптимизатор AdamW (он лучше обычного Adam за счет правильной регуляризации весов)
optimizer = optim.AdamW(model.parameters(), lr=3e-4)

# 3. Функция потерь (игнорируем пустышки)
criterion = nn.CrossEntropyLoss(ignore_index=trg_pad_idx)

In [32]:
# Создаем фейковый батч (2 предложения)
# src: 2 предложения по 10 слов
src_data = torch.randint(1, src_vocab_size, (2, 10))
# trg: 2 предложения по 12 слов (перевод длиннее)
trg_data = torch.randint(1, trg_vocab_size, (2, 12))

# Делаем прогон (Forward Pass)
out = model(src_data, trg_data)

print(f"Форма выхода: {out.shape}")
# Ожидаемый результат: torch.Size([2, 12, 5000])
# [Размер батча, Длина перевода, Размер словаря]

Форма выхода: torch.Size([2, 12, 5000])


In [33]:
import torch

# Количество эпох
epochs = 20

# Переводим модель в режим обучения (важно для слоев нормализации!)
model.train()

print("Начинаем обучение...")

for epoch in range(epochs):
    # В реальности здесь был бы цикл по батчам из DataLoader
    # for batch in dataloader:
    #     src = batch.src
    #     trg = batch.trg

    # Мы используем наши фейковые данные:
    src = src_data
    trg = trg_data

    # =========================================================
    # ТРЮК СО СДВИГОМ (Teacher Forcing)
    # =========================================================
    # Вход для Декодера: берем всё предложение, КРОМЕ последнего слова (токена <END>)
    # Пример: [START] J'aime les chats
    trg_input = trg[:, :-1]

    # Ожидаемый ответ: берем всё предложение, КРОМЕ первого слова (токена <START>)
    # Пример: J'aime les chats [END]
    trg_expected = trg[:, 1:]


    # =========================================================
    # 5 СВЯЩЕННЫХ ШАГОВ PYTORCH
    # =========================================================

    # Шаг 1. Обнуляем градиенты с прошлого шага (чтобы они не накапливались)
    optimizer.zero_grad()

    # Шаг 2. Forward Pass (Прогон)
    # Передаем в модель оригинал и сдвинутый перевод
    output = model(src, trg_input)

    # Шаг 3. Вычисляем Loss
    # CrossEntropyLoss требует специфической формы тензоров:
    # output: сплющиваем батчи и слова в одну длинную колбасу -> [batch_size * seq_len, vocab_size]
    # trg_expected: сплющиваем в одномерный массив правильных ответов -> [batch_size * seq_len]
    output = output.reshape(-1, output.shape[2])
    trg_expected = trg_expected.reshape(-1)

    loss = criterion(output, trg_expected)

    # Шаг 4. Backward Pass (Вычисляем, кто из весов виноват в ошибке)
    loss.backward()

    # Шаг 5. Делаем шаг оптимизатора (Обновляем веса)
    optimizer.step()

    print(f"Эпоха [{epoch+1}/{epochs}] | Ошибка (Loss): {loss.item():.4f}")

print("Обучение завершено!")

Начинаем обучение...
Эпоха [1/20] | Ошибка (Loss): 8.7750
Эпоха [2/20] | Ошибка (Loss): 7.2361
Эпоха [3/20] | Ошибка (Loss): 6.1596
Эпоха [4/20] | Ошибка (Loss): 5.3581
Эпоха [5/20] | Ошибка (Loss): 4.6835
Эпоха [6/20] | Ошибка (Loss): 4.0755
Эпоха [7/20] | Ошибка (Loss): 3.5122
Эпоха [8/20] | Ошибка (Loss): 2.9621
Эпоха [9/20] | Ошибка (Loss): 2.4414
Эпоха [10/20] | Ошибка (Loss): 1.9895
Эпоха [11/20] | Ошибка (Loss): 1.6204
Эпоха [12/20] | Ошибка (Loss): 1.3264
Эпоха [13/20] | Ошибка (Loss): 1.0914
Эпоха [14/20] | Ошибка (Loss): 0.9018
Эпоха [15/20] | Ошибка (Loss): 0.7485
Эпоха [16/20] | Ошибка (Loss): 0.6256
Эпоха [17/20] | Ошибка (Loss): 0.5283
Эпоха [18/20] | Ошибка (Loss): 0.4517
Эпоха [19/20] | Ошибка (Loss): 0.3914
Эпоха [20/20] | Ошибка (Loss): 0.3437
Обучение завершено!


In [34]:
import torch

batch_size = 20
src_vocab_size = 5000
trg_vocab_size = 5000
pad_idx = 0
start_idx = 1
end_idx = 2

src_lens = torch.randint(3, 15, (batch_size,))
trg_lens = torch.randint(4, 18, (batch_size,)) # Перевод

max_src_len = src_lens.max() + 2
max_trg_len = trg_lens.max() + 2

src_data = torch.randint(1, src_vocab_size, (batch_size, max_src_len))
trg_data = torch.randint(1, trg_vocab_size, (batch_size, max_trg_len))
src_data[:, 0] = start_idx
trg_data[:, 0] = start_idx
src_data[torch.arange(batch_size), src_lens + 1] = end_idx
trg_data[torch.arange(batch_size), trg_lens + 1] = end_idx

src_positions = torch.arange(max_src_len)
src_mask = src_positions <= (src_lens + 1).unsqueeze(1)
trg_mask = torch.arange(max_trg_len) <= (trg_lens + 1).unsqueeze(1)

# 5. Заменяем на pad_idx (0) все места, где маска = False (знак ~ означает логическое НЕ)
src_data.masked_fill_(~src_mask, pad_idx)
trg_data.masked_fill_(~trg_mask, pad_idx)

print("Размер сгенерированного src:", src_data.shape)
print("Длины первых 3 предложений (src):", src_lens[:3].tolist())
print("Первые 3 предложения (src_data):\n", src_data[:3])

Размер сгенерированного src: torch.Size([20, 16])
Длины первых 3 предложений (src): [4, 9, 12]
Первые 3 предложения (src_data):
 tensor([[   1,  190,  320,  832,  645,    2,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0],
        [   1, 4889, 2643, 2030, 4094, 2939, 2853, 4396,  435, 2489,    2,    0,
            0,    0,    0,    0],
        [   1, 2710, 3498, 4976, 1832, 2096, 3837, 4990, 1188, 3700, 3028,  671,
         1196,    2,    0,    0]])


In [35]:
src_positions

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])

In [36]:
src_lens.unsqueeze(1)

tensor([[ 4],
        [ 9],
        [12],
        [ 5],
        [12],
        [ 4],
        [13],
        [ 4],
        [ 8],
        [14],
        [11],
        [10],
        [14],
        [13],
        [14],
        [ 8],
        [ 8],
        [ 5],
        [ 9],
        [ 8]])

In [37]:
src_mask

tensor([[ True,  True,  True,  True,  True,  True, False, False, False, False,
         False, False, False, False, False, False],
        [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True, False, False, False, False, False],
        [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True, False, False],
        [ True,  True,  True,  True,  True,  True,  True, False, False, False,
         False, False, False, False, False, False],
        [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True, False, False],
        [ True,  True,  True,  True,  True,  True, False, False, False, False,
         False, False, False, False, False, False],
        [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True,  True, False, False, False, False,
    

In [38]:
src_data

tensor([[   1,  190,  320,  832,  645,    2,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0],
        [   1, 4889, 2643, 2030, 4094, 2939, 2853, 4396,  435, 2489,    2,    0,
            0,    0,    0,    0],
        [   1, 2710, 3498, 4976, 1832, 2096, 3837, 4990, 1188, 3700, 3028,  671,
         1196,    2,    0,    0],
        [   1,  643, 3696, 3480,  406, 3159,    2,    0,    0,    0,    0,    0,
            0,    0,    0,    0],
        [   1, 4396, 1207, 4294, 3433, 3868, 2259, 1431, 3127, 2971, 4105,  222,
         4724,    2,    0,    0],
        [   1, 1632, 3240, 4062, 3150,    2,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0],
        [   1, 2191,  295, 4666, 4496, 3592, 2072, 1457, 3237, 3928, 2333, 3043,
         3680, 4345,    2,    0],
        [   1, 2841, 3012, 1485, 4875,    2,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0],
        [   1, 4528, 1741, 4607,  199, 3499,   16, 3817, 3138,    2,    0,    0,

In [39]:
src_data.shape

torch.Size([20, 16])

In [40]:
trg_data.shape

torch.Size([20, 19])

In [42]:
epochs = 100
max_length = int(max(max_src_len, max_trg_len))
embed_size = 512
num_heads = 8
num_layers = 6
model = FullTransformer(src_vocab_size=src_vocab_size, trg_vocab_size=trg_vocab_size, max_length=max(max_src_len, max_trg_len), embed_size=embed_size, num_heads=num_heads, num_layers=num_layers, src_pad_idx=pad_idx, trg_pad_idx=pad_idx, forward_expansion=4)
model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss(ignore_index=trg_pad_idx)
for epoch in range(epochs):
    src = src_data
    trg = trg_data
    trg_input = trg[:, :-1]
    trg_expected = trg[:, 1:]
    optimizer.zero_grad()
    output = model(src, trg_input)
    output = output.reshape(-1, output.shape[-1])
    trg_expected = trg_expected.reshape(-1)
    loss = criterion(output, trg_expected)
    loss.backward()
    optimizer.step()

    print(f"Эпоха [{epoch+1}/{epochs}] | Ошибка (Loss): {loss.item():.4f}")

Эпоха [1/100] | Ошибка (Loss): 8.6818
Эпоха [2/100] | Ошибка (Loss): 7.6178
Эпоха [3/100] | Ошибка (Loss): 7.5033
Эпоха [4/100] | Ошибка (Loss): 6.6738
Эпоха [5/100] | Ошибка (Loss): 5.8432
Эпоха [6/100] | Ошибка (Loss): 5.2662
Эпоха [7/100] | Ошибка (Loss): 4.5123
Эпоха [8/100] | Ошибка (Loss): 4.0321
Эпоха [9/100] | Ошибка (Loss): 3.5781
Эпоха [10/100] | Ошибка (Loss): 3.0013
Эпоха [11/100] | Ошибка (Loss): 2.5567
Эпоха [12/100] | Ошибка (Loss): 2.0920
Эпоха [13/100] | Ошибка (Loss): 1.7138
Эпоха [14/100] | Ошибка (Loss): 1.3781
Эпоха [15/100] | Ошибка (Loss): 1.0994
Эпоха [16/100] | Ошибка (Loss): 0.8877
Эпоха [17/100] | Ошибка (Loss): 0.7192
Эпоха [18/100] | Ошибка (Loss): 0.5898
Эпоха [19/100] | Ошибка (Loss): 0.4936
Эпоха [20/100] | Ошибка (Loss): 0.4197
Эпоха [21/100] | Ошибка (Loss): 0.3596
Эпоха [22/100] | Ошибка (Loss): 0.3106
Эпоха [23/100] | Ошибка (Loss): 0.2663
Эпоха [24/100] | Ошибка (Loss): 0.2238
Эпоха [25/100] | Ошибка (Loss): 0.1825
Эпоха [26/100] | Ошибка (Loss): 0.

In [43]:
def generate_translation(model, src_tensor, start_token_idx, end_token_idx, max_length = 50):
    model.eval()
    trg_indices = [start_token_idx]
    for i in range(max_length):
        trg_tensor = torch.LongTensor(trg_indices).unsqueeze(0).to(src_tensor.device)
        with torch.no_grad():
            output = model(src_tensor, trg_tensor)
        last_word_prediction = output[:, -1, :]
        best_guess = last_word_prediction.argmax(dim=-1).item()
        trg_indices.append(best_guess)
        if best_guess == end_token_idx:
            break
    return trg_indices


In [44]:
test_src = src_data[0].unsqueeze(0)
translated_indices = generate_translation(model, test_src, start_idx, end_idx)
print("Оригинал (индексы):", test_src[0].tolist())
print("Перевод (индексы):", translated_indices)


Оригинал (индексы): [1, 190, 320, 832, 645, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Перевод (индексы): [1, 4437, 1799, 4226, 2888, 841, 1312, 63, 1038, 4854, 3039, 3230, 2901, 1253, 4825, 1162, 4582, 2]
